# QQ PLOTS 
 
Quantile–quantile plots and genomic inflation values for summary statistics from set-based tests using SAIGE-GENE+ are calculated for each maximum minor allele frequency threshold and test type, including burden, SKAT, and SKAT-O.
- Table 1 : Genomic inflation 
- Plot 1 : Quantile-quantile plot 

In [ ]:
# ========================================================================================================
# INSTALL PACKAGES  
# ========================================================================================================

# Install packages 
if (!requireNamespace("flextable", quietly = TRUE)) {
  install.packages("flextable")
}

if (!requireNamespace("officer", quietly = TRUE)) {
  install.packages("officer")
}

if (!requireNamespace("janitor", quietly = TRUE)) {
  install.packages("janitor")
}

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("biomaRt")

install.packages("xlsx")

install.packages("paletteer")

# ========================================================================================================
# LOAD PACKAGES  
# ========================================================================================================

# Load the packages
library(tidyverse)
library(ggplot2)
library(data.table)
library(purrr)

# Install these packages 
library(janitor)
library(flextable)
library(officer)
library(biomaRt)
library(patchwork)
library(xlsx)

In [ ]:
# ========================================================================================================
# FUNCTIONS
# ========================================================================================================

# Function: Formats the gene set-based rare variant assoication test results 
process_results <- function(results) { 
    results %>% 
    filter(Group != "Cauchy") %>% 
    # inner_join(gene_symbols, by = "Region") %>% 
    mutate(max_MAF = as.character(max_MAF)) %>% 
    mutate(negative_log_p_values = -log10(Pvalue)) %>% 
    # mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
    mutate(Chromosome = as.numeric(Chromosome)) %>% 
    mutate(Region = as.factor(Region)) %>% 
    mutate(Region_factor = fct_reorder(Region, Chromosome))
  }

# HIGH CONFIDENCE ANNOATIONS 

In [ ]:
# ========================================================================================================
# FORMAT SUMMARY STATISTICS 
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

files <- c()
stats <- c() 

system("mkdir -p local_dir") 
phenotypes <- c("aml_phenotype",
                "aml_biosample_age", 
                "aml_diagnosis_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", pheno))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            pheno, 
            "_high_confidence_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_high_confidence_chr", chr)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(pheno,"_high_confidence")]] <- stat
}

# ========================================================================================================
# FORMAT SUMMARY STATISTICS 
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

system("mkdir -p local_dir") 
phenotypes <- c("biosample_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test: ", pheno))
        print(paste0("Loading chromosome: ", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/biosample_collection_check/EUR_",
            pheno, 
            "_high_confidence_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_high_confidence_chr", chr)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(pheno,"_high_confidence")]] <- stat
}

# ========================================================================================================
# FORMAT SUMMARY STATISTICS 
# ========================================================================================================

# Load data from workspace bucket  
system("mkdir -p local_dir") 
covariates <- c("_deprivation_index",
                "_deprivation")

for (cov in covariates){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", cov))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            "aml_phenotype", 
            "_high_confidence_chr",
            chr, 
            cov, 
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       "aml_phenotype",
                       "_high_confidence_chr", chr, cov)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(cov,"_high_confidence")]] <- stat
}


# SYNONYMOUS ANNOATIONS

In [ ]:
# ========================================================================================================
# FORMAT SUMMARY STATISTICS 
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

system("mkdir -p local_dir") 
phenotypes <- c("aml_phenotype",
                "aml_biosample_age", 
                "aml_diagnosis_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", pheno))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            pheno, 
            "_synonymous_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_synonymous_chr", chr)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(pheno,"_synonymous")]] <- stat
}

# ========================================================================================================
# FORMAT SUMMARY STATISTICS 
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

system("mkdir -p local_dir") 

phenotypes <- c("biosample_age")

for (pheno in phenotypes){
    
    files <- list()   # reset per phenotype
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test: ", pheno))
        print(paste0("Loading chromosome: ", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/biosample_collection_check/EUR_",
            pheno, 
            "_synonymous_chr",
            chr,
            " local_dir/"
        )

        system(cmd)

        path <- paste0(
            "local_dir/EUR_",
            pheno,
            "_synonymous_chr", chr
        )

        # If multiple files exist (common with SAIGE output)
        if (dir.exists(path)) {
            file_list <- list.files(path, full.names = TRUE)
            f <- rbindlist(lapply(file_list, fread), fill = TRUE)
        } else {
            f <- fread(path)
        }
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- rbindlist(files, fill = TRUE)
    stats[[paste0(pheno, "_synonymous")]] <- stat
}

# ========================================================================================================
# FORMAT SUMMARY STATISTICS 
# ========================================================================================================

# Load data from workspace bucket  
system("mkdir -p local_dir") 
covariates <- c("_deprivation_index",
                "_deprivation")

for (cov in covariates){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", cov))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            "aml_phenotype", 
            "_synonymous_chr",
            chr, 
            cov, 
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       "aml_phenotype",
                       "_synonymous_chr", chr, cov)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(cov,"_synonymous")]] <- stat
}



# GENOMIC INFLATION 

In [ ]:
# ========================================================================================================
# RENAME ANALYSIS IDENTIFIER 
# ========================================================================================================

# Clean name of list 
clean_names <- function(x) {
  x <- gsub("_", " ", x)
  
  sapply(strsplit(x, " "), function(w) {
    paste(toupper(substring(w, 1, 1)), substring(w, 2), sep = "", collapse = " ")
  })
}

new_names <- c(
  "aml_phenotype_high_confidence", 
  "aml_biosample_age_high_confidence", 
  "aml_diagnosis_age_high_confidence", 
  "biosample_age_components_high_confidence", 
  "aml_deprivation_index_high_confidence",
  "aml_deprivation_components_high_confidence",
  
  "aml_phenotype_synonymous", 
  "aml_biosample_age_synonymous", 
  "aml_diagnosis_age_synonymous", 
  "biosample_age_components_synonymous", 
  "aml_deprivation_index_synonymous",
  "aml_deprivation_components_synonymous"
)

names(stats) <- clean_names(new_names)

# ========================================================================================================
# FORMAT SUMMARY STATISTICS FOR DOWNLOAD 
# ========================================================================================================

df_list <- stats

genomic_inflation_list <- list()

for (df_name in names(df_list)) {
    
    df <- df_list[[df_name]]  # get the current data frame
    # Transform the full dataset for plotting
    genomic_inflation <- 
    df %>%
    filter(!is.na(Pvalue) & Pvalue > 0 & Pvalue <= 1) %>%
    pivot_longer(
    cols = starts_with("Pvalue"),
    names_to = "Statistical_Test",
    names_prefix = "Pvalue_",
    values_to = "P_value",
    values_drop_na = TRUE
    ) %>%
     mutate(
         Statistical_Test = ifelse(Statistical_Test == "Pvalue", "SKAT-O", Statistical_Test)
     ) %>%
    mutate(max_MAF = case_when(
    max_MAF == "0.01"   ~ "1 x 10⁻²",
    max_MAF == "0.001"  ~ "1 x 10⁻³",
    max_MAF == "1e-04"  ~ "1 x 10⁻⁴",
    TRUE ~ max_MAF)) %>%
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(chisq = qchisq(P_value,1,lower.tail=FALSE)) %>% 
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(lambda = (median(chisq) / qchisq(0.5,1)),
           lambda_0.01 = quantile(chisq, probs = 0.99) / qchisq(0.99, df = 1)) %>% 
    distinct(Statistical_Test,max_MAF,lambda,lambda_0.01) %>% 
    rename("Statistical Test" = "Statistical_Test", 
         "Maximum MAF" = "max_MAF", 
         "Lambda" = "lambda",
         "Lambda 0.01" = "lambda_0.01") 

    genomic_inflation$Analysis <- df_name
    genomic_inflation_list[[df_name]] <- genomic_inflation

}

# ========================================================================================================
# SAVE RESULTS 
# ========================================================================================================

# Combine genomic inflation data frames 
genomic_inflation_df <- do.call(rbind, genomic_inflation_list)

genomic_inflation_df <- genomic_inflation_df[, c(
  "Analysis", 
  "Statistical Test", 
  "Maximum MAF", 
  "Lambda", 
  "Lambda 0.01"
)]

# Create column description sheet
column_descriptions <- data.frame(
  Column = c(
    "Analysis",
    "Statistical Test",
    "Maximum MAF",
    "Lambda",
    "Lambda 0.01"
  ),
  Description = c(
    "Name of the analysis performed",
    "Statistical test used in the analysis",
    "Maximum minor allele frequency threshold used for variant inclusion",
    "Genomic inflation factor (lambda GC)",
    "Genomic inflation factor for the top 1% of p-values"
  )
)

# Write Excel with multiple sheets
library(openxlsx)

wb <- createWorkbook()

# First sheet
addWorksheet(wb, "Genomic Inflation")
writeData(wb, "Genomic Inflation", genomic_inflation_df)

# Create the second sheet BEFORE writing to it
addWorksheet(wb, "Column Descriptions")

writeData(
  wb,
  "Column Descriptions",
  column_descriptions
)

saveWorkbook(
  wb,
  "genomic_inflation.xlsx",
  overwrite = TRUE
)